# LocUp — Train MobileBERT and export to TFLite

This notebook **fine-tunes `google/mobilebert-uncased`** for 5 epochs on a 250-post hyperlocal-post dataset, then **converts it to TFLite with dynamic-range quantisation** and **exports four assets** that the Android app consumes directly.

## What you get

After running all cells you'll have a single `.zip` containing:

| File | Size | Purpose |
|---|---|---|
| `locup_text_classifier.tflite` | ~25 MB | MobileBERT fine-tune, INT8 quantised. Loaded by `TfliteTextClassifier` via `FileChannel.map()`. |
| `vocab.txt` | ~500 KB | BERT tokenizer vocab (one token per line, line number = id). Used by `BertTokenizer` on-device. |
| `labels.txt` | < 1 KB | Class names, one per line, in model output order. |
| `max_seq_len.txt` | < 1 KB | A single integer, e.g. `64`. The Android side reads this on startup. |

## How to run

1. Open in Colab (T4 GPU recommended; ~5 min total).  
   <https://colab.research.google.com/github/your-name/towntalk/blob/main/tools/train_locup_classifier.ipynb>
2. **Runtime → Change runtime type → T4 GPU**.
3. Run cells in order. The last cell downloads a `.zip` of the four assets.
4. Unzip into the repo's `app/src/main/assets/` folder (overwriting the old word-embedding `locup_text_classifier.tflite` and deleting the old `vocab.json`).
5. `./gradlew assembleDebug` and the APK ships with MobileBERT.

## Why this exists

The repo deliberately ships a **real transformer on-device** rather than a custom Keras model. The intent is to showcase the "fine-tune in Colab, convert to TFLite, run on Android with a hand-rolled tokenizer" pipeline end-to-end. The Kotlin `BertTokenizer` is hand-rolled and must mirror the Python `AutoTokenizer` rules exactly — see the test at `app/src/test/.../TfliteTextClassifierTokenizeTest.kt`.

## Tokenizer parity

Critical: the **exact same `vocab.txt`** is loaded by Python training and by the Android app. If you regenerate the model with a different tokenizer (e.g. `cased` instead of `uncased`), the Kotlin `BertTokenizer` will produce wrong ids. Keep both sides in sync.

## 1. Install dependencies

Stable as of 2024. `transformers` brings `AutoTokenizer` and `TFAutoModel`. `tensorflow` is needed for the Keras training loop and `TFLiteConverter`. No `tflite-model-maker` / `mediapipe-model-maker` — those paths were broken on current Colab.

In [ ]:
!pip install -q transformers tensorflow

## 2. Mount Drive (so we can save the assets folder)

This is optional — if you skip it, the notebook will save assets to `/tmp` and the download at the end will still work. Mount Drive if you want to keep the assets folder around between runs.

In [ ]:
import pathlib
ASSETS_DIR = pathlib.Path('/content/locup_assets')
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Assets will be written to: {ASSETS_DIR}')

## 3. Configuration

These constants **must** match what the Android app expects:

- `MODEL_NAME` — controls both the base architecture and the vocab. The Kotlin `BertTokenizer` is written for the BERT-uncased tokenization scheme (lowercase + accent strip + WordPiece). If you change this, change both sides.
- `MAX_SEQ_LEN` — written to `max_seq_len.txt`, read by the Kotlin classifier at startup.
- `LABELS` — order matters. `labels.txt` is read line-by-line by the Kotlin side and zipped with the model's output indices.

In [ ]:
import os, random, re, unicodedata, pathlib
import numpy as np
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel

MODEL_NAME = "google/mobilebert-uncased"
MAX_SEQ_LEN = 64
LABELS = ["Emergency", "Traffic", "Event", "Civic", "General"]
NUM_LABELS = len(LABELS)
SEED = 7

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'Model: {MODEL_NAME}')
print(f'Max sequence length: {MAX_SEQ_LEN}')
print(f'Labels: {LABELS}')

## 4. The 250-post training set

50 short realistic posts per category. Same dataset as in `tools/train_locup_classifier.py`. No API calls, no external datasets — verifiable, reproducible.

In [ ]:
POSTS = [
    # Emergency (50)
    ("fire on 5th avenue, big smoke, evacuate now", "Emergency"),
    ("car accident at the main junction, need ambulance", "Emergency"),
    ("waterlogging in basement, pump truck please", "Emergency"),
    ("gas leak smell near school, fire dept on the way", "Emergency"),
    ("downed power line sparking on sidewalk, do not approach", "Emergency"),
    ("building collapse on mg road, rescue team needed", "Emergency"),
    ("two-wheeler skid near market, person bleeding", "Emergency"),
    ("short circuit in flat, smoke from switchboard", "Emergency"),
    ("river overflowing, sandbagging at the bridge", "Emergency"),
    ("tree fell on car after storm, driver trapped", "Emergency"),
    ("stranger lurking near kids park, pls watch out", "Emergency"),
    ("medical emergency at the metro station", "Emergency"),
    ("gas cylinder leaked in apartment, fire brigade called", "Emergency"),
    ("car into shop window, glass everywhere", "Emergency"),
    ("child missing in the colony, please help search", "Emergency"),
    ("wall collapse during rains, debris on the road", "Emergency"),
    ("open manhole on the main road, dangerous at night", "Emergency"),
    ("snake spotted in the parking lot, call forest dept", "Emergency"),
    ("stray dog bite near the school gate", "Emergency"),
    ("burning smell from the transformer, lights flickering", "Emergency"),
    ("floodwater entering houses, need rescue boats", "Emergency"),
    ("live wire on the footpath, two people shocked", "Emergency"),
    ("major pileup on the highway, blocking both lanes", "Emergency"),
    ("fire in the high-rise, sprinklers going off", "Emergency"),
    ("ambulance stuck in traffic, please give way", "Emergency"),
    ("robbery at the atm, two suspects on bike", "Emergency"),
    ("building on fire near the hospital", "Emergency"),
    ("flood relief camp accepting donations at the school", "Emergency"),
    ("sewage overflow on the street, unhygienic and unsafe", "Emergency"),
    ("broken glass on the cycle track, cyclist injured", "Emergency"),
    ("gas leak in the restaurant, customers evacuated", "Emergency"),
    ("power cuts everywhere since morning, no eta", "Emergency"),
    ("house flooded up to the doorstep, pls help", "Emergency"),
    ("accident near the school gate, kid hurt", "Emergency"),
    ("smoke from the park, looks like a fire", "Emergency"),
    ("bridge closed, structural damage after the quake", "Emergency"),
    ("two cars collided, airbags deployed, police needed", "Emergency"),
    ("fall from terrace, person unresponsive", "Emergency"),
    ("fire alarm in the mall, please evacuate orderly", "Emergency"),
    ("wells overflowing, kids playing nearby, dangerous", "Emergency"),
    ("rescue operation at the lake, two swimmers missing", "Emergency"),
    ("eye injury from cracker, going to hospital", "Emergency"),
    ("fire in the timber yard, smoke visible from km away", "Emergency"),
    ("stranger trying door handles, please alert", "Emergency"),
    ("road washout after the rain, crater in the middle", "Emergency"),
    ("petrol pump flooded, no fuel available", "Emergency"),
    ("kid fell into the open drain, rescued", "Emergency"),
    ("fire in the electrical room, building evacuated", "Emergency"),
    ("missing elderly person last seen at the temple", "Emergency"),
    ("two-wheeler accident, helmet saved the rider", "Emergency"),
    # Traffic (50)
    ("traffic jam on the highway, moving at snail pace", "Traffic"),
    ("road closed for the marathon, take the bypass", "Traffic"),
    ("signal not working at the main crossing, chaos", "Traffic"),
    ("lane closed for pipe laying, expect delays", "Traffic"),
    ("heavy congestion near the market, leave early", "Traffic"),
    ("traffic police diversion at the school, follow signs", "Traffic"),
    ("vehicle breakdown in the tunnel, single lane", "Traffic"),
    ("roadblock at the bridge, use the alternative route", "Traffic"),
    ("construction work on the flyover, expect 30 min delay", "Traffic"),
    ("metro work blocking the right lane, slow traffic", "Traffic"),
    ("traffic piling up at the toll plaza, cash lanes slow", "Traffic"),
    ("waterlogging under the bridge, slow traffic", "Traffic"),
    ("long queue at the petrol pump, two pumps down", "Traffic"),
    ("protest march on the main road, full diversion", "Traffic"),
    ("container truck breakdown, lane blocked", "Traffic"),
    ("traffic signal cycling green thrice then red, faulty", "Traffic"),
    ("parking full at the mall, do not drive in", "Traffic"),
    ("buses diverted due to the rally, no service on route 12", "Traffic"),
    ("road repair at the station, single lane traffic", "Traffic"),
    ("crane removing the overturned truck, expect 2 hours", "Traffic"),
    ("school pickup causing traffic everywhere at 3pm", "Traffic"),
    ("delivery truck double parked, blocking the lane", "Traffic"),
    ("signal jumped, two wheeler hit, slight jam", "Traffic"),
    ("diversion via 2nd main, two-way traffic on a one-way", "Traffic"),
    ("bumper to bumper from the tech park to the bridge", "Traffic"),
    ("auto strike today, public transport limited", "Traffic"),
    ("metro station crowd spilling onto the road", "Traffic"),
    ("tree branch on the road, slow traffic", "Traffic"),
    ("loud honking at the crossing, classic weekday", "Traffic"),
    ("container fell off the truck, road closed", "Traffic"),
    ("weekend cricket match, road near the ground blocked", "Traffic"),
    ("diversion for the marathon, use the ring road", "Traffic"),
    ("pedestrian crossing blocked by hawkers, traffic slow", "Traffic"),
    ("traffic pile-up at the malfunctioning boom barrier", "Traffic"),
    ("school zone, low speed, expect slow movement", "Traffic"),
    ("work-from-home advisory due to heavy traffic", "Traffic"),
    ("traffic jammed at the new signal, sync not done", "Traffic"),
    ("road widened but no new lane markings, drivers confused", "Traffic"),
    ("no parking on the main road, towing in progress", "Traffic"),
    ("protest rally entering the area, slow traffic", "Traffic"),
    ("traffic from the airport to the city moving fast", "Traffic"),
    ("metro line extension work, two lanes taken", "Traffic"),
    ("overnight road work, single lane now", "Traffic"),
    ("bumper entry restricted today, only residents", "Traffic"),
    ("train timing changed, station road unusually empty", "Traffic"),
    ("flyover opened last night, smooth traffic now", "Traffic"),
    ("bus caught fire near the depot, traffic diverted", "Traffic"),
    ("sensor-based signal not detecting two-wheelers", "Traffic"),
    ("nighttime road closure for the film shoot", "Traffic"),
    ("road caved in, only one lane open", "Traffic"),
    # Event (50)
    ("community yoga session at the park this sunday", "Event"),
    ("weekend farmers market in the society", "Event"),
    ("diwali mela at the club, families welcome", "Event"),
    ("garage sale outside flat 12, 10am to 4pm", "Event"),
    ("garba practice at the community hall, join in", "Event"),
    ("local marathon route through the colony, sunday 6am", "Event"),
    ("blood donation camp at the school, register now", "Event"),
    ("poetry meetup at the cafe, open mic", "Event"),
    ("music concert at the amphitheatre, free entry", "Event"),
    ("food festival at the lakeside, this weekend", "Event"),
    ("holi celebration at the club, colors provided", "Event"),
    ("auction at the community center, vintage items", "Event"),
    ("kite flying contest on the terrace, trophies", "Event"),
    ("festival procession through the main road, 5pm", "Event"),
    ("cricket match between blocks, saturday morning", "Event"),
    ("parenting workshop at the library, free", "Event"),
    ("dance class registrations open at the studio", "Event"),
    ("plant sale at the nursery, weekend only", "Event"),
    ("carrom tournament at the sports complex", "Event"),
    ("trekking group meeting point: parking lot, 6am", "Event"),
    ("holi get-together at the secretary's house", "Event"),
    ("book fair at the school ground for 3 days", "Event"),
    ("open mic night at the brewery, friday", "Event"),
    ("society ganesh chaturthi celebration, all welcome", "Event"),
    ("republic day parade practice, school ground", "Event"),
    ("annual day at the community hall, invite attached", "Event"),
    ("weekend film screening at the amphitheatre", "Event"),
    ("drawing competition for kids at the club", "Event"),
    ("navratri garba at the society ground, 8pm", "Event"),
    ("carols by the lake on christmas eve", "Event"),
    ("new year countdown at the rooftop cafe", "Event"),
    ("valentine's day couples dance at the lounge", "Event"),
    ("society bbq and pool party this saturday", "Event"),
    ("tabla concert at the temple courtyard, evening", "Event"),
    ("photo walk meetup at the metro station, 7am", "Event"),
    ("zumba class launch at the park, free trial", "Event"),
    ("sketching class for beginners at the studio", "Event"),
    ("pet adoption drive at the mall, sunday", "Event"),
    ("community kitchen serving lunch at the temple", "Event"),
    ("open library at the clubhouse, 10am to 6pm", "Event"),
    ("inter-block cricket tournament, finals saturday", "Event"),
    ("hackathon at the tech park, 48 hours", "Event"),
    ("society culturals on the 5th, performances open", "Event"),
    ("monsoon food festival at the lake, enter free", "Event"),
    ("kids science fair at the school, open to all", "Event"),
    ("wine and cheese tasting at the lounge, 7pm", "Event"),
    ("comedy night at the brewery, two shows", "Event"),
    ("early morning cycling meetup at the park", "Event"),
    ("weekend mahjong at the senior citizens club", "Event"),
    ("lantern festival at the lake, evening", "Event"),
    # Civic (50)
    ("garbage pile near the bin, not collected for days", "Civic"),
    ("streetlight not working on the main road, dangerous", "Civic"),
    ("pothole on the main road, two-wheelers skidding", "Civic"),
    ("sewage overflow on the pavement, smells awful", "Civic"),
    ("dump yard overflowing, dogs scattering trash", "Civic"),
    ("open drain on the footpath, kids playing nearby", "Civic"),
    ("broken footpath near the school, accessibility issue", "Civic"),
    ("no water supply in the block since morning", "Civic"),
    ("low water pressure, top floors not getting any", "Civic"),
    ("dirty water coming from the tap, not drinkable", "Civic"),
    ("storm water drain blocked, water stagnating", "Civic"),
    ("public toilet closed for two weeks, no maintenance", "Civic"),
    ("streetlight pole leaning, can fall anytime", "Civic"),
    ("speed breaker unmarked, drivers cant see it", "Civic"),
    ("encroachment on the footpath, no walking space", "Civic"),
    ("mosquito breeding in the stagnant water at the park", "Civic"),
    ("garbage truck missed our street last week", "Civic"),
    ("dhobi ghat drainage choked, water on the road", "Civic"),
    ("illegal parking on the service road, no action", "Civic"),
    ("nala cleaning overdue, blocking the storm water", "Civic"),
    ("zebra crossing paint faded, no visibility", "Civic"),
    ("bus stop bench broken, no shelter from rain", "Civic"),
    ("electric pole wires hanging low, dangerous", "Civic"),
    ("water meter reading wrong, bill inflated", "Civic"),
    ("no dustbins on the main road, people littering", "Civic"),
    ("footpath dug up by the cable operator, no repair", "Civic"),
    ("manhole cover missing, dangerous at night", "Civic"),
    ("public garden not maintained, dead plants", "Civic"),
    ("fallen tree branch not cleared, road narrow", "Civic"),
    ("semi-permanent hawker stalls blocking the lane", "Civic"),
    ("water pump not working in the apartment", "Civic"),
    ("drain smell from the neighbour, not cleaned", "Civic"),
    ("parking lot full of potholes, cars getting damaged", "Civic"),
    ("bus shelter glass broken, no safety", "Civic"),
    ("pavement tiles loose, trip hazard", "Civic"),
    ("dumping of construction debris at the vacant plot", "Civic"),
    ("sewage manhole cover broken, stink all over", "Civic"),
    ("water tanker not coming for the past 3 days", "Civic"),
    ("public taps leaking, wasting water", "Civic"),
    ("kala azar mosquito breeding in the stagnant water", "Civic"),
    ("hand pump broken, no water for the slum", "Civic"),
    ("bus stop on the highway without shelter", "Civic"),
    ("roadside trees not trimmed, blocking the signages", "Civic"),
    ("drain cleaning request ignored for months", "Civic"),
    ("construction dust on the road, no water sprinkling", "Civic"),
    ("pedestrian signal not working at the junction", "Civic"),
    ("waste segregation bins removed by someone", "Civic"),
    ("streetlight timing wrong, dark till late morning", "Civic"),
    ("stray cattle on the road, no action from ward", "Civic"),
    ("parking chaos outside the metro station, no markings", "Civic"),
    # General (50)
    ("hello, anyone from the b block online?", "General"),
    ("hi all, new to the colony, drop a hi", "General"),
    ("thanks for the help yesterday, much appreciated", "General"),
    ("lost my dog near the park, brown labrador, please call", "General"),
    ("found a set of keys near the temple, contact me", "General"),
    ("shoutout to the watchman, very helpful", "General"),
    ("can anyone recommend a good plumber in the area?", "General"),
    ("maid available for part-time work, contact me", "General"),
    ("anyone selling a used washing machine, working condition", "General"),
    ("pet sitting available next week, experienced", "General"),
    ("hello, does anyone know the timings for the post office?", "General"),
    ("hi, looking for a roommate for a 2bhk", "General"),
    ("thanks to the sweeper for cleaning the staircase", "General"),
    ("lost my wallet near the bus stop, please return", "General"),
    ("found a child's cycle near the park, contact me", "General"),
    ("anyone interested in a book exchange at the club", "General"),
    ("recommend a good tiffin service for the area", "General"),
    ("share your evening walks photos in the group", "General"),
    ("good morning, beautiful sunrise from my balcony", "General"),
    ("good night all, see you tomorrow", "General"),
    ("happy birthday to the friendly neighborhood uncle", "General"),
    ("anybody from the c block, lets catch up", "General"),
    ("hi, looking for a carpool to the tech park", "General"),
    ("tiffin service recommendations, only veg", "General"),
    ("anyone selling a study table, used ok", "General"),
    ("hi all, what is the wifi password for the club", "General"),
    ("share your monsoon pictures in the group", "General"),
    ("good morning, tea at the tapri is back", "General"),
    ("hi, which grocery store is open late night", "General"),
    ("recommend a good paediatrician in the area", "General"),
    ("sharing a recipe for soft idlis, let me know results", "General"),
    ("happy diwali everyone, stay safe", "General"),
    ("happy new year, best wishes from our block", "General"),
    ("hi all, sharing a heritage walk pdf, useful", "General"),
    ("anyone going to the airport tomorrow, can share", "General"),
    ("found a pair of spectacles near the library", "General"),
    ("hi all, anyone selling a treadmill, working", "General"),
    ("greetings, looking for a yoga instructor for home", "General"),
    ("happy holi, share your photos after playing", "General"),
    ("sharing a poem i wrote on the colony", "General"),
    ("hi all, any local cricket coach available", "General"),
    ("recommend a good salon for haircut", "General"),
    ("happy independence day, flag hoisting at the club", "General"),
    ("found my missing cat, thanks for the help", "General"),
    ("hi all, looking for a used bicycle for my kid", "General"),
    ("recommend a good electrician, work in the kitchen", "General"),
    ("share your terrace garden photos in the group", "General"),
    ("hi all, any plumber available on sunday", "General"),
    ("shoutout to the milk booth, never late", "General"),
    ("hi all, sharing the colony directory pdf", "General"),
    ("thanks for the quick help with the broken pipe", "General"),
]

assert len(POSTS) == 250, f'expected 250 posts, got {len(POSTS)}'
for label in LABELS:
    count = sum(1 for _, l in POSTS if l == label)
    assert count == 50, f'label {label} has {count} posts, expected 50'
print(f'Dataset OK: {len(POSTS)} posts, {NUM_LABELS} classes, 50 each.')

## 5. Tokenization helpers (Python reference for the Kotlin tokenizer)

These two functions reproduce the *exact* tokenisation rules the Kotlin `BertTokenizer` uses:
- `basic_tokenize` — NFKD + accent strip + lowercase + the standard BERT basic-tokenizer regex.
- `wordpiece_tokenize` — greedy longest-match-first WordPiece with `##` continuation prefixes.

If you ever change these, change the Kotlin side at the same time. The unit tests on `app/src/test/.../BertTokenizerTest.kt` are the contract.

In [ ]:
def basic_tokenize(text):
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    pat = r"'(?:s|ve|re|ll|t|m|d)|[a-z]+|[0-9]+|[^\sa-z0-9]+|\s+"
    return [m.group(0).strip() for m in re.finditer(pat, text) if m.group(0).strip()]

def wordpiece_tokenize(token, vocab, unk_token="[UNK]"):
    out, start, n = [], 0, len(token)
    while start < n:
        end, cur = n, None
        while start < end:
            sub = token[start:end] if start == 0 else "##" + token[start:end]
            if sub in vocab:
                cur = sub; break
            end -= 1
        if cur is None:
            return [unk_token]
        out.append(cur); start = end
    return out

def encode(text, vocab, max_seq_len, cls_id, sep_id, pad_id, unk_id):
    pieces = []
    for tok in basic_tokenize(text):
        pieces.extend(wordpiece_tokenize(tok, vocab))
    pieces = pieces[:max_seq_len - 2]
    ids = [cls_id] + [vocab.get(w, unk_id) for w in pieces] + [sep_id]
    while len(ids) < max_seq_len:
        ids.append(pad_id)
    mask = [1 if ids[i] != pad_id else 0 for i in range(max_seq_len)]
    return ids[:max_seq_len], mask[:max_seq_len]

print("basic('Fire on MG Road!') =", basic_tokenize("Fire on MG Road!"))

## 6. Load MobileBERT + tokenizer

First run downloads ~100 MB of weights. Subsequent runs hit the HF cache.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
base = TFAutoModel.from_pretrained(MODEL_NAME)
print(f'Loaded {MODEL_NAME}.')
print(f'Vocab size: {len(tokenizer.vocab):,}')
print(f'CLS={tokenizer.cls_token_id} SEP={tokenizer.sep_token_id} PAD={tokenizer.pad_token_id} UNK={tokenizer.unk_token_id}')

## 7. Build the classification head

We take the `[CLS]` token's hidden state, dropout, and a 5-class softmax. The output tensor is named `classifier` so the Flutter side maps to it via `mapOf("classifier" to output)` in `TfliteTextClassifier.kt`.

In [ ]:
input_ids = tf.keras.Input(shape=(MAX_SEQ_LEN,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(MAX_SEQ_LEN,), dtype=tf.int32, name="attention_mask")
token_type_ids = tf.keras.Input(shape=(MAX_SEQ_LEN,), dtype=tf.int32, name="token_type_ids")
outputs = base(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
cls = outputs.last_hidden_state[:, 0, :]
x = tf.keras.layers.Dropout(0.1)(cls)
logits = tf.keras.layers.Dense(NUM_LABELS, activation="softmax", name="classifier")(x)
model = tf.keras.Model(inputs=[input_ids, attention_mask, token_type_ids], outputs=logits)
model.summary()

## 8. Train

5 epochs at lr=2e-5, batch size 16. With the 250-post dataset this is ~5 min on a T4 GPU.

In [ ]:
def prepare_dataset(texts, labels):
    enc = tokenizer(texts, max_length=MAX_SEQ_LEN, padding="max_length",
                    truncation=True, return_tensors="tf")
    label_to_idx = {l: i for i, l in enumerate(LABELS)}
    y = np.array([label_to_idx[l] for l in labels], dtype=np.int32)
    return enc["input_ids"], enc["attention_mask"], enc["token_type_ids"], y

rng = np.random.default_rng(SEED)
idx = np.arange(len(POSTS)); rng.shuffle(idx)
split = int(0.9 * len(POSTS))
train_idx, val_idx = idx[:split], idx[split:]
texts = [p[0] for p in POSTS]; labels = [p[1] for p in POSTS]
train_texts = [texts[i] for i in train_idx]; train_labels = [labels[i] for i in train_idx]
val_texts   = [texts[i] for i in val_idx];   val_labels   = [labels[i] for i in val_idx]

x_train = prepare_dataset(train_texts, train_labels)
x_val   = prepare_dataset(val_texts, val_labels)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x=[x_train[0], x_train[1], x_train[2]], y=x_train[3],
          validation_data=([x_val[0], x_val[1], x_val[2]], x_val[3]),
          epochs=5, batch_size=16, verbose=2)

## 9. Sanity-check accuracy on the full set

We expect >90% on this small synthetic dataset. If it's lower, the head weights didn't train — usually means the learning rate is wrong or the dataset is too small.

In [ ]:
x_all = prepare_dataset(texts, labels)
preds = model.predict([x_all[0], x_all[1], x_all[2]], verbose=0).argmax(axis=1)
acc = (preds == x_all[3]).mean()
print(f'Full-set accuracy: {acc:.3f}')
for i, label in enumerate(LABELS):
    mask = (x_all[3] == i)
    if mask.sum() > 0:
        print(f'  {label:>10s}: {((preds[mask] == i).mean()):.3f}  ({mask.sum()} posts)')

## 10. Convert to TFLite with dynamic-range quantisation

`Optimize.DEFAULT` is dynamic-range quant. It quantises weights to int8 but keeps activations in float, which is the standard trade-off for transformer models — ~4× size reduction, minimal accuracy loss, no calibration dataset needed.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
print(f'TFLite model size: {len(tflite_model):,} bytes ({len(tflite_model) / 1024 / 1024:.1f} MB)')

## 11. Export the four assets

These four files go into `app/src/main/assets/`. They are the **exact same** artefacts the Android side consumes — same `vocab.txt` (one token per line, line number = id), same `labels.txt` order, same `max_seq_len.txt`.

In [ ]:
model_path = ASSETS_DIR / "locup_text_classifier.tflite"
model_path.write_bytes(tflite_model)
print(f'wrote {model_path} ({model_path.stat().st_size:,} bytes)')

vocab_path = ASSETS_DIR / "vocab.txt"
vocab_path.write_text(tokenizer.vocab, encoding="utf-8")
print(f'wrote {vocab_path} ({vocab_path.stat().st_size:,} bytes, {len(tokenizer.vocab):,} tokens)')

labels_path = ASSETS_DIR / "labels.txt"
labels_path.write_text("\n".join(LABELS) + "\n", encoding="utf-8")
print(f'wrote {labels_path} ({labels_path.stat().st_size:,} bytes)')

seq_len_path = ASSETS_DIR / "max_seq_len.txt"
seq_len_path.write_text(str(MAX_SEQ_LEN) + "\n", encoding="utf-8")
print(f'wrote {seq_len_path} ({seq_len_path.stat().st_size:,} bytes)')

## 12. Download the assets

Zips the four files into a single download so you can grab them in one go.

In [ ]:
import shutil
zip_path = pathlib.Path('/content/locup_assets.zip')
shutil.make_archive(str(ASSETS_DIR), 'zip', root_dir=str(ASSETS_DIR.parent), base_dir=ASSETS_DIR.name)
print(f'Zipped: {zip_path} ({zip_path.stat().st_size:,} bytes)')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Not running in Colab — pick up the zip manually at', zip_path)

## 13. Drop the assets into the APK

Unzip `locup_assets.zip` into your repo:

```
unzip locup_assets.zip -d app/src/main/assets/
```

This **replaces** the old `locup_text_classifier.tflite` (the 579 KB word-embedding model) and **deletes** the old `vocab.json` (the new tokenizer expects `vocab.txt`). The old `locup_text_classifier.tflite` should be removed first if present:

```
rm -f app/src/main/assets/vocab.json
```

After the swap, the four files in `app/src/main/assets/` should be:

```
locup_text_classifier.tflite   ~25 MB  ← the MobileBERT fine-tune
vocab.txt                       ~500 KB  ← BERT vocab
labels.txt                      < 1 KB  ← class names
max_seq_len.txt                 < 1 KB  ← "64" (or whatever MAX_SEQ_LEN you set)
```

Build and install:

```
./gradlew assembleDebug
adb install -r app/build/outputs/apk/debug/app-debug.apk
```

Open the app → tap **New post** → type `fire on 5th avenue`. The bottom-sheet banner should turn **green** to signal the real MobileBERT path is active. logcat will show:

```
PostClassifierRepo: classify("fire on 5th avenue") -> Emergency (0.97) usedRealModel=true
TfliteTextClassifier: Interpreter loaded — inputs: [input_ids, attention_mask, token_type_ids], output: classifier
BertTokenizer: BertTokenizer ready: vocabSize=30522 maxSeqLen=64
```

You're done. The app is now running a real MobileBERT fine-tune on-device.

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
Banner stays amber | `vocab.json` still present, or `vocab.txt` missing | Delete `vocab.json`, ensure `vocab.txt` exists |
All `0.2` outputs | Vague symptom of mismatched vocab | Check that `vocab.txt` was generated by THIS notebook, not the old training script |
`Interpreter.run` exception | Wrong tensor names | Look at logcat for the actual input/output names printed by `TfliteTextClassifier.tryLoad()` |
OOM during training | Batch size too large | Drop to `batch_size=8` or `MAX_SEQ_LEN=32` |
Low accuracy on real posts | Synthetic dataset is too thin | Edit the `POSTS` list to add real examples, retrain |

## When to regenerate

Re-run this notebook when you want to:
- Add real posts to the dataset (edit `POSTS`, retrain).
- Switch to `cased` MobileBERT (or a different family — change `MODEL_NAME`, the Kotlin tokenizer, and the tests together).
- Try a different `MAX_SEQ_LEN` (32, 96, 128 — for shorter/longer posts).
- Try full int8 quantisation (replace `Optimize.DEFAULT` with `Optimize.DEFAULT` + a representative calibration dataset).